In [1]:

import pandas as pd
import geopandas as gpd
from pathlib import Path

# Calculate regional losses excel based on schakels and their regions

In [2]:
output_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Losses_Analysis\recompute_regionale_losses")

In [109]:

import geopandas as gpd
from pathlib import Path

# Load data
schakels_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Losses_Analysis\recompute_regionale_losses\aggregated_loss_per_nws_code.gpkg")
regions_path  = Path(r"P:\bovenregionale-stresstest-hwn\Data\Hazard_maps\Areas.gpkg")

schakels = gpd.read_file(schakels_path)
regions  = gpd.read_file(regions_path)

# Ensure same CRS
if schakels.crs != regions.crs:
    regions = regions.to_crs(schakels.crs)


In [110]:

import geopandas as gpd
from pathlib import Path


output_path = Path(
    r"P:\bovenregionale-stresstest-hwn\Analysis\Losses_Analysis"
    r"\recompute_regionale_losses"
)


# ------------------------------------------------------------------
# Clean schema to avoid FID errors
# ------------------------------------------------------------------
schakels = schakels.reset_index(drop=True)
regions  = regions.reset_index(drop=True)

for col in ['fid', 'FID', 'id']:
    if col in schakels.columns:
        schakels = schakels.drop(columns=col)
    if col in regions.columns:
        regions = regions.drop(columns=col)

# ------------------------------------------------------------------
# Ensure same CRS (must be projected, meters)
# ------------------------------------------------------------------
if schakels.crs != regions.crs:
    regions = regions.to_crs(schakels.crs)

# ------------------------------------------------------------------
# Spatial intersection
# ------------------------------------------------------------------
regions_sub = regions[['name', 'geometry']]

intersections = gpd.overlay(
    schakels[['NWSCODE', 'geometry']],
    regions_sub,
    how='intersection',
    keep_geom_type=False
)

# ------------------------------------------------------------------
# Compute overlap size
# Use length for line geometries
# ------------------------------------------------------------------
intersections['overlap_size'] = intersections.length

# ------------------------------------------------------------------
# Determine primary region (largest overlap)
# ------------------------------------------------------------------
primary_region = (
    intersections
    .sort_values(['NWSCODE', 'overlap_size'], ascending=[True, False])
    .groupby('NWSCODE', as_index=False)
    .first()[['NWSCODE', 'name']]
    .rename(columns={'name': 'primary_region'})
)

# ------------------------------------------------------------------
# Collect all overlapping regions
# ------------------------------------------------------------------
regions_overlap = (
    intersections
    .groupby('NWSCODE')['name']
    .apply(lambda x: sorted(set(x)))
    .reset_index()
    .rename(columns={'name': 'regions_overlap'})
)

# ------------------------------------------------------------------
# Join results back to schakels
# ------------------------------------------------------------------
schakels = schakels.merge(
    regions_overlap,
    on='NWSCODE',
    how='left'
)

schakels = schakels.merge(
    primary_region,
    on='NWSCODE',
    how='left'
)

# ------------------------------------------------------------------
# Convert list → string for GeoPackage compatibility
# ------------------------------------------------------------------
schakels['regions_overlap'] = schakels['regions_overlap'].apply(
    lambda x: ', '.join(x) if isinstance(x, list) else x
)

# ------------------------------------------------------------------
# Save output
# ------------------------------------------------------------------
schakels.to_file(
    output_path / "Losses_per_schakel_with_regions.gpkg",
    layer="schakels_regions",
    driver="GPKG"
)


CPLE_AppDefinedError: b'sqlite3_exec(CREATE TRIGGER "trigger_delete_feature_count_schakels_regions" AFTER DELETE ON "schakels_regions" BEGIN UPDATE gpkg_ogr_contents SET feature_count = feature_count - 1 WHERE lower(table_name) = lower(\'schakels_regions\'); END;) failed: unable to open database file'

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b'sqlite3_exec(CREATE TRIGGER "trigger_delete_feature_count_schakels_regions" AFTER DELETE ON "schakels_regions" BEGIN UPDATE gpkg_ogr_contents SET feature_count = feature_count - 1 WHERE lower(table_name) = lower(\'schakels_regions\'); END;) failed: unable to open database file'


In [111]:
regions_overlap

,NWSCODE,regions_overlap
0,001-0010,[ARK-NZK]
1,001-0020,[ARK-NZK]
2,001-0030,[ARK-NZK]
3,001-0040,"[ARK-NZK, Vallei en Veluwe]"
4,001-0055,[Vallei en Veluwe]
...,...,...
239,079-0015,[Limburg]
240,099-0010,[ARK-NZK]
241,200-0010,[ARK-NZK]
242,200-0020,[ARK-NZK]


In [112]:

import geopandas as gpd
from pathlib import Path

# --------------------------------------------------
# Intersect schakels with regions
# --------------------------------------------------
intersections = gpd.overlay(
    schakels[['NWSCODE', 'geometry']],
    regions[['name', 'geometry']],
    how='intersection'
)

# --------------------------------------------------
# Length of overlap per segment
# --------------------------------------------------
intersections['overlap_m'] = intersections.length

# --------------------------------------------------
# Total schakel length
# --------------------------------------------------
schakel_length = (
    schakels[['NWSCODE', 'geometry']]
    .assign(schakel_len_m=lambda g: g.length)
    .drop(columns='geometry')
)

# --------------------------------------------------
# Sum overlap per (schakel, region)
# --------------------------------------------------
weights = (
    intersections
    .groupby(['NWSCODE', 'name'], as_index=False)
    .agg({'overlap_m': 'sum'})
)

# --------------------------------------------------
# Add schakel length
# --------------------------------------------------
weights = weights.merge(
    schakel_length,
    on='NWSCODE',
    how='left'
)

# --------------------------------------------------
# Percentage overlap (weights sum to 1 per schakel)
# --------------------------------------------------
weights['pct_overlap'] = (
    weights['overlap_m'] /
    weights.groupby('NWSCODE')['overlap_m'].transform('sum')
)



In [113]:
weights

,NWSCODE,name,overlap_m,schakel_len_m,pct_overlap
0,001-0010,ARK-NZK,13155.120090,13155.120090,1.000000
1,001-0020,ARK-NZK,35878.157477,35878.157477,1.000000
2,001-0030,ARK-NZK,37073.333387,37073.333387,1.000000
3,001-0040,ARK-NZK,0.131989,59160.980911,0.000002
4,001-0040,Vallei en Veluwe,59160.848922,59160.980911,0.999998
...,...,...,...,...,...
291,099-0010,ARK-NZK,21377.799849,21377.799849,1.000000
292,200-0010,ARK-NZK,14479.035269,14479.035269,1.000000
293,200-0020,ARK-NZK,14633.841557,14633.841557,1.000000
294,915-0010,Noord-Westelijke Delta,4482.689639,7655.568620,0.585547


In [74]:
schakels.columns

Index(['NWSCODE', 'Length_schakel', 'NWSNAAM', 'VHLH_AL_E_WR', 'VHLH_L1_E_WR',
       'VHLH_L2_E_WR', 'VHLH_L3_E_WR', 'F_EV2_ma', 'F_EV1_me', 'AL_E_WR',
       'getroffen_personen_perdag', 'getroffen_vracht_L2+L3_E_WR_perdag',
       'vracht_op_schakel_totaal', 'personen_op_schakel_totaal',
       'VOT_L1_gemiddeld', 'VOT_L2L3_gemiddeld', 'VOT_total', 'geometry',
       'regions_overlap', 'primary_region'],
      dtype='object')

✅ Does your code produce redistributed region totals?
Yes, your region_loss_summary is redistributed because you:

joined weights (with pct_overlap) to schakels,
multiplied each loss column by pct_overlap,
grouped by name (region).

So the region totals are split by overlap as intended. ✅

In [149]:

# Join overlap weights to schakels
schakels_w = schakels.merge(
    weights[['NWSCODE', 'name', 'pct_overlap']],
    on='NWSCODE',
    how='left'
)

agg_columns = [
    'VHLH_AL_E_WR', 'VHLH_L1_E_WR', 'VHLH_L2_E_WR', 'VHLH_L3_E_WR',
    'AL_E_WR', 'VOT_L1_gemiddeld', 'VOT_L2L3_gemiddeld', 'VOT_total',
    'getroffen_personen_perdag', 'getroffen_vracht_L2+L3_E_WR_perdag'
]

for col in agg_columns:
    schakels_w[col] = schakels_w[col] * schakels_w['pct_overlap']

mask_vhlh_positive = schakels_w['VHLH_AL_E_WR'] > 0


region_loss_summary = (
    schakels_w[mask_vhlh_positive]
    .groupby('name')[agg_columns]
    .sum()
    .reset_index()
    .round(0)
)


# Filter alleen positieve waarden
mask_pos = schakels["F_EV2_ma"] > 0

# Gemiddelde per regio (primary_region) op basis van positieve waarden
avg_per_region = (
    schakels.loc[mask_pos]
    .groupby("primary_region")["F_EV2_ma"]
    .mean()
    .rename("avg_duration")
    .reset_index()
)

avg_per_region



,primary_region,avg_duration
0,ARK-NZK,590.455701
1,Achterhoek,127.375000
2,Brabantse Delta,216.538462
3,Friesland,240.000000
4,Groningen en NO-Drenthe,240.000000
5,Limburg,66.001041
6,Noord-Brabant Oost,297.110713
7,Noord-Westelijke Delta,188.571429
8,Overijsselse Vecht,346.000000
9,Rivierenland,134.241760


In [ ]:

# mask = (schakels["primary_region"] == "Rivierenland") & (schakels["F_EV2_ma"] > 0)
# average_duration = schakels.loc[mask, "F_EV2_ma"].mean()
# average_duration


134.24176025390625

In [150]:
schakels_w.to_file(output_path / "schakels_met_gewicht.gpkg")

CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET min_x = 26744.54030000045, min_y = 307279.8260000012, max_x = 278021.4070999994, max_y = 606213.6999999992 WHERE lower(table_name) = lower('schakels_met_gewicht') AND Lower(data_type) = 'features') failed: disk I/O error"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET min_x = 26744.54030000045, min_y = 307279.8260000012, max_x = 278021.4070999994, max_y = 606213.6999999992 WHERE lower(table_name) = lower('schakels_met_gewicht') AND Lower(data_type) = 'features') failed: disk I/O error"


In [155]:
region_loss_summary = region_loss_summary.merge(
    avg_per_region,
    left_on='name', 
    right_on='primary_region',
    how='inner'
)

region_loss_summary.to_excel(
    output_path / "Region_Loss_Summary_filtered2.xlsx",
    index=False
)


In [156]:
region_loss_summary

,name,VHLH_AL_E_WR,VHLH_L1_E_WR,VHLH_L2_E_WR,VHLH_L3_E_WR,AL_E_WR,VOT_L1_gemiddeld,VOT_L2L3_gemiddeld,VOT_total,getroffen_personen_perdag,getroffen_vracht_L2+L3_E_WR_perdag,primary_region,avg_duration
0,ARK-NZK,29435318.0,25980008.0,1897227.0,1558105.0,1399716.0,270711685.0,218031438.0,488743123.0,1092415.0,134491.0,ARK-NZK,590.455701
1,Achterhoek,2038282.0,1636227.0,208015.0,194046.0,357380.0,17049483.0,25370052.0,42419535.0,320641.0,70326.0,Achterhoek,127.375000
2,Brabantse Delta,7179580.0,5781670.0,662246.0,735700.0,847950.0,60245006.0,88210375.0,148455382.0,677248.0,149482.0,Brabantse Delta,216.538462
3,Friesland,2002790.0,1742817.0,141242.0,118730.0,203539.0,18160154.0,16404264.0,34564418.0,194668.0,25996.0,Friesland,240.000000
4,Groningen en NO-Drenthe,2121235.0,1796469.0,181178.0,143588.0,228370.0,18719206.0,20492745.0,39211951.0,198008.0,32094.0,Groningen en NO-Drenthe,240.000000
5,Limburg,2421080.0,1851791.0,271029.0,298264.0,894157.0,19295660.0,35922402.0,55218062.0,774017.0,197485.0,Limburg,66.001041
6,Noord-Brabant Oost,12011656.0,9870628.0,1026160.0,1114936.0,1094613.0,102851942.0,135103180.0,237955122.0,999624.0,196015.0,Noord-Brabant Oost,297.110713
7,Noord-Westelijke Delta,9513286.0,8528266.0,471043.0,513996.0,1239104.0,88864531.0,62155984.0,151020515.0,1080207.0,120519.0,Noord-Westelijke Delta,188.571429
8,Overijsselse Vecht,6476757.0,5218430.0,647628.0,610730.0,466985.0,54376044.0,79402365.0,133778409.0,403559.0,88066.0,Overijsselse Vecht,346.000000
9,Rivierenland,615597.0,491808.0,67011.0,56777.0,180057.0,5124640.0,7811021.0,12935661.0,100432.0,24012.0,Rivierenland,134.241760


In [106]:
region_loss_summary

,name,VHLH_AL_E_WR,VHLH_L1_E_WR,VHLH_L2_E_WR,VHLH_L3_E_WR,AL_E_WR,VOT_L1_gemiddeld,VOT_L2L3_gemiddeld,VOT_total,getroffen_personen_perdag,getroffen_vracht_L2+L3_E_WR_perdag,avg_F_EV2_ma_x,avg_F_EV2_ma_y
0,ARK-NZK,29435318.0,25980008.0,1897227.0,1558105.0,1399716.0,270711685.0,218031438.0,488743123.0,1092415.0,134491.0,587.18,587.18
1,Achterhoek,2038282.0,1636227.0,208015.0,194046.0,357380.0,17049483.0,25370052.0,42419535.0,320641.0,70326.0,128.10,128.10
2,Brabantse Delta,7179580.0,5781670.0,662246.0,735700.0,847950.0,60245006.0,88210375.0,148455382.0,677248.0,149482.0,220.47,220.47
3,Friesland,2002790.0,1742817.0,141242.0,118730.0,203539.0,18160154.0,16404264.0,34564418.0,194668.0,25996.0,240.00,240.00
4,Groningen en NO-Drenthe,2121235.0,1796469.0,181178.0,143588.0,228370.0,18719206.0,20492745.0,39211951.0,198008.0,32094.0,240.00,240.00
5,Limburg,2421080.0,1851791.0,271029.0,298264.0,894157.0,19295660.0,35922402.0,55218062.0,774017.0,197485.0,78.32,78.32
6,Noord-Brabant Oost,12011656.0,9870628.0,1026160.0,1114936.0,1094613.0,102851942.0,135103180.0,237955122.0,999624.0,196015.0,299.67,299.67
7,Noord-Westelijke Delta,9513286.0,8528266.0,471043.0,513996.0,1239104.0,88864531.0,62155984.0,151020515.0,1080207.0,120519.0,201.60,201.60
8,Overijsselse Vecht,6476757.0,5218430.0,647628.0,610730.0,466985.0,54376044.0,79402365.0,133778409.0,403559.0,88066.0,331.25,331.25
9,Rivierenland,615597.0,491808.0,67011.0,56777.0,180057.0,5124640.0,7811021.0,12935661.0,100432.0,24012.0,156.53,156.53
